# 第2回 線形回帰で学ぶ汎化性能とリークの再現（解答例・教員用）


---


## 前回からの接続


第1回では、気温帯・年・天気で分けた平均で「明日何台か」に答えました。毎日平均と答えると外れ幅の平均は1582台、気温帯×年で分けると762台でした。ただ、その答えは気温帯の境目で跳ぶガタガタの段でした。しかも、細かく分けるほど外れ幅は縮みました。でもそれは、平均を作るのに使った日で採点していたからでした。

今日は、この段を1本の式にします。そして、その式がまだ来ていない日にも当たるのかを確かめます。


---


## 今日の分析目標


気温を入れると台数が出てくる1本の式を立て、その式がまだ来ていない日にどれだけ当たるかを測る。

1日ぶんの計算から始め、731日ぶん・総当たり・正規方程式と積み上げて「良い直線」を選び、最後に`LinearRegression`の1行と比べます。そのあと、データを分けて本当の実力を測ります。最後の「目標に答えられたか」で振り返りましょう。


### 場面

場面は前回と同じです。あなたは自転車シェアの運営者で、毎晩、明日に向けて自転車を何台用意するかを決めます。

決めたいのは、気温などを入れると明日の台数が出てくる式です。その式が、まだ来ていない日にどれだけ当たるのかも知っておきたいところです。前回の似た日の平均は、境目で答えが跳びました。しかも作るのに使った日で採点していたので、本当の当たり具合が分かりませんでした。


### 手元のデータ

前回と同じ、ある自転車シェアの2011年1月1日から2012年12月31日までの記録です（UCI Machine Learning RepositoryのBike Sharing Dataset）。1行が1日で、731日ぶんあります。行は日付の順に、過去から未来へ並んでいます。毎晩当てたかったのは、その日の台数（`台数`）です。

前回、この表から2つのことが分かりました。気温が高い日ほど台数が多く、ただし暑すぎると頭打ちになります。同じ気温でも、2年目のほうが多くなります。今日はおもに、`気温`（0〜1に縮めた値。1が約41℃）と`年`、そして`台数`を使います。

表をもう一度よく見ると、気になる列が2つあります。1つは`一般客`と`会員`の列で、足すと731日すべてでちょうど`台数`になります。明日の一般客と会員の数は、明日が来るまで分かりません。もう1つは`日付`で、表は過去から未来へ並んでいます。明日を当てたいのに、未来の日を見て学んでしまわないか、気をつける必要があります。この2つの列には、今日の後半で戻ってきます。


### 最後に出す答え

まだ来ていない日で測った式の当たり具合と、その式で見積もった明日の台数です。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 1日ぶんの予測・外れ・外れの二乗を計算し、`for`文で731日ぶん足し上げられる
- `for`文と`if`文の総当たりで二乗和の谷底を探し、最小二乗法（least squares）の考え方を説明できる
- 二乗和を微分して正規方程式を導き、numpyの行列の計算（`@`・`.T`・`np.linalg.solve`）でコードにできる。`LinearRegression`の答えと一致することを確かめられる
- R²を「毎日平均と答える基準から、外れの二乗和をどれだけ減らせたか」として計算できる
- データを訓練用とテスト用に分け、作るのに使ったデータでの成績と、まだ見ていないデータでの成績（汎化性能）を比べられる
- 答えに近い情報が手がかりに漏れ込む「リーク」を再現し、テストの成績が良すぎるときに疑える
- 式を曲げすぎたときの過学習（overfitting）を、訓練とテストのR²の差で説明できる


---


## 0. 設定


### 最初に実行する：道具を呼び出す設定のセル

- 役割：このノートで使う道具（ライブラリ）を呼び出し、データを読み込んで、列名を日本語に付け替える
- 入力：なし
- 出力：「セットアップ完了」の表示と、列名が日本語の表`df`

今日は、表のpandasと図のmatplotlibに加えて、数の並びをまとめて計算するnumpyと、回帰の道具が入ったscikit-learn（`sklearn`）を使います。列名の付け替えは、第1回の1節と同じ前処理です（`season`→`季節`、`temp`→`気温`、`cnt`→`台数`など）。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

# 図中の日本語が □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ

# 自転車シェアの日次データ（1日1行）を読み込む
# Colab で使うときは公開データの URL に変える（配布時に設定）
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
df = pd.read_csv(f'{DATA_DIR}/bike_day.csv')
# 列名を日本語に付け替える（第1回の1節と同じ前処理）
ja_names = {'instant': '番号', 'dteday': '日付', 'season': '季節',
            'yr': '年', 'mnth': '月', 'holiday': '祝日',
            'weekday': '曜日', 'workingday': '平日',
            'weathersit': '天気', 'temp': '気温', 'atemp': '体感気温',
            'hum': '湿度', 'windspeed': '風速', 'casual': '一般客',
            'registered': '会員', 'cnt': '台数'}
df = df.rename(columns=ja_names)
print('セットアップ完了')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は ② の解析が中心です。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i == 2
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


### 今日の道筋

段を1本の式にし、その式がまだ来ていない日にも当たるかを確かめます。次の5つの小さな問いに、順に答えていきます。

1. 直線の良さを、どう測る？（1〜3節）
2. いちばん良い直線は？（4〜5節）
3. まだ見ていない日に、当たる？（6節）
4. 曲げれば、もっと当たる？（7節）
5. 明日は、何台？（8節）


---


## 1. 列を、数の並びとして取り出す


今日は、前回の段を1本の式に置き換えます。式を立てるには、気温と台数を「1日ずつの数の並び」として扱えると便利です。


### おさらい：列を選ぶ

表から1つの列を取り出すには、`[ ]`に列名を入れます。

```python
df['台数']        # 台数の列だけ
```

取り出したものは、日付の番号つきの列（Series）です。


### お手本：列を配列にして、1日目を見る

- 役割：気温と台数の列を、数の並び（配列）として取り出し、1日目の値を見る
- 入力：表`df`
- 出力：配列`x`（気温）と`y`（台数）

`.values`を付けると、列がnumpyの配列（array）になります。配列は`x[0]`のように番号で1つずつ取り出せます。番号は0から数えます。


In [ ]:
x = df['気温'].values     # 気温（731日ぶん）
y = df['台数'].values      # 台数（731日ぶん）
print(len(x))             # いくつ並んでいるか
print(x[0])               # 1日目の気温
print(y[0])               # 1日目の台数


731個の数が並んでいます。1日目は、気温0.344（41を掛けると約14.1℃）で、台数は985台でした。


---


## 2. 1日目を、直線で予測すると？


1節で、1日目の気温と台数を取り出せました。前回の段の代わりに、いちばん単純な式を考えます。直線です。

$$
\text{台数} \approx a \times \text{気温} + b
$$

記号で書くと、$i$日目の気温（0〜1に縮めた値）を$x_i$、台数を$y_i$、直線による予測を$\hat{y}_i$（ワイ・ハット。ハットは「予測」の印）として、

$$
\hat{y}_i = a\,x_i + b \qquad (i = 1, 2, \dots, 731)
$$

`気温`は、実際の気温（℃）を41で割って0〜1に縮めた値です。実際の気温を$t_i$とすると$x_i = t_i/41$なので、$\hat{y}_i = \frac{a}{41}\,t_i + b$と書けます。a（傾き）は、`気温`が1増えたときに台数がいくら増えるか。1℃あたりの増え方は$a/41$です（a=6600なら1℃で約160台）。b（切片）は、`気温`が0のときの台数です。aとbの2つの数を決めれば、式が完成します。このように直線（1次式）の式で当てることを、線形回帰（linear regression）と呼びます。でも、aとbの組み合わせは無数にあります。どれが「良い直線」なのでしょうか。まずは適当なaとbで、1日目だけを予測してみます。


### お手本：1日目の予測と外れ

- 役割：a=9000・b=0の直線で1日目を予測し、外れ（実際 − 予測）とその二乗を出す
- 入力：`x[0]`・`y[0]`と、a・b
- 出力：予測・外れ・外れの二乗

予測が実際からどれだけずれたか、$e_i = y_i - \hat{y}_i$を残差（residual）と呼びます。外れは多すぎても少なすぎても困るので、符号を消すために二乗します。二乗にはもう1つ良さがあります。aやbを動かすと、二乗の和はなめらかな谷の形になります。そのため、あとで見るように谷底を式一発で求められるのです（4節・5節）。大きく外れた日ほど重く数えられるという性質もあります。


In [ ]:
a = 9000
b = 0
pred = a * x[0] + b       # 1日目の予測
print(pred)
err = y[0] - pred         # 外れ（実際 − 予測）
print(err)
print(err ** 2)           # 外れの二乗


予測は約3098台。実際は985台なので、外れは −2112.5台（多く見積もりすぎた）、その二乗は約446万です。


### TODO①：2日目と3日目も、同じように計算する

上のお手本のセルをコピーして、`x[0]`・`y[0]`を2日目（`[1]`）と3日目（`[2]`）に変え、予測・外れ・外れの二乗を表示してください。


In [ ]:
# TODO: 上のセルをコピーして、[0] を [1]（2日目）と

# 解答例①：2日目と3日目
pred = a * x[1] + b
print(pred, y[1] - pred, (y[1] - pred) ** 2)
pred = a * x[2] + b
print(pred, y[2] - pred, (y[2] - pred) ** 2)


2日目の外れは −2470.3、3日目は −418.3です。同じことを731日ぶん書くのは大変です。次の節で、繰り返しを書く方法を使います。


---


## 3. 731日ぶんを、足し上げる


直線の良さを測るには、1日だけでなく、全部の日の外れを見る必要があります。外れの二乗を全日で足した二乗和を、直線の物差しにします。小さいほど、よく当たる直線です。

$$
S(a, b) = \sum_{i=1}^{731} \bigl(y_i - (a\,x_i + b)\bigr)^2
$$

$\sum$は、$i=1$から731までを足し上げる記号です。aとbを決めると二乗和が1つに決まるので、$S(a, b)$と書きます。下の`for`文は、この$\sum$の1項ずつを足していきます。


### お手本：for文で足し上げる

- 役割：731日ぶんの外れの二乗を、1日ずつ足し上げる
- 入力：配列`x`・`y`と、a・b
- 出力：二乗和`total`

`for i in range(len(x)):`は、「`i`を0, 1, 2, …, 730と変えながら、下の字下げした行を繰り返す」という書き方です（for文）。字下げ（行の頭のすきま）が、繰り返す範囲の目印です。`total = total + …`は、「いまの合計に足して、合計を更新する」という意味です。


In [ ]:
total = 0
for i in range(len(x)):
    pred = a * x[i] + b                  # その日の予測
    total = total + (y[i] - pred) ** 2   # 外れの二乗を足す
print(total)


二乗和は約18.0億です。2節で計算した1日目の446万は、この中の1日ぶんです。


### 変えてみる：numpyで、まとめて計算する

numpyの配列は、`for`文を書かなくてもまとめて計算できます。`a * x + b`と書くと、731日ぶんの予測が一度に出ます。`.sum()`は全部を足すメソッドです。

`residual[:3]`は、配列の番号0・1・2の3つを取り出す書き方です（スライス）。`[始め:終わり]`の形で、始めを省くと0から、終わりの番号は含みません。


In [ ]:
residual = y - (a * x + b)       # 731日ぶんの外れを、まとめて
print(residual[:3])              # 最初の3日ぶん
print((residual ** 2).sum())     # 二乗して、全部足す


最初の3日ぶんは、2節とTODO①で1日ずつ計算した値と同じです。二乗和も`for`文の答えと一致しました。`for`文で書いたことを、numpyは1行でやってくれていたのです。


---


## 4. いちばん良いaとbは？　総当たりで探す


二乗和がいちばん小さくなるaとbを探します。まずは素朴に、aをいろいろ試してみます。


### お手本：aを100ずつ試す

- 役割：bを1200に固定し、aを5000〜8000まで100ずつ変えて、二乗和がいちばん小さいaを探す
- 入力：配列`x`・`y`
- 出力：いちばん良いaと、そのときの二乗和

`range(5000, 8001, 100)`のように数を3つ渡すと、`range(始め, 終わり, 刻み)`の意味になり、始めから刻みずつ増えていきます。終わりの数は含まないので、8000まで入れたいときは8001と書きます。まず、小さな範囲で確かめてみましょう。


In [ ]:
for a in range(5000, 5301, 100):
    print(a)          # 5000, 5100, 5200, 5300 と表示される


`if`文は「条件が成り立つときだけ、下の字下げした行を実行する」書き方です（if文）。ここでは「これまでの最小より小さければ、覚え直す」に使います。`None`は「まだ何も入っていない」という印で、1回目は必ず覚えます。`A or B`は「AとBのどちらかが成り立てば成り立つ」という意味です。1回目は`best_sse is None`で成り立ち、2回目からは`s < best_sse`で決まります。


In [ ]:
best_a = None
best_sse = None
for a in range(5000, 8001, 100):              # 5000, 5100, …, 8000
    s = ((y - (a * x + 1200)) ** 2).sum()     # この a の二乗和
    if best_sse is None or s < best_sse:      # 小さければ覚え直す
        best_a = a
        best_sse = s
print(best_a, best_sse)


a=6700のとき、二乗和が最小になりました。aを動かしたときの二乗和を図にすると、谷底が1つだけのU字になります。


In [ ]:
# @title 図：a を動かしたときの二乗和
a_list = list(range(5000, 8001, 100))
sse_list = [((y - (a * x + 1200)) ** 2).sum() for a in a_list]
plt.plot(a_list, sse_list, 'o-', color='#0066cc')
plt.axvline(best_a, color='#e63946', ls='--')
plt.xlabel('傾き a（b は 1200 に固定）'); plt.ylabel('二乗和'); plt.show()


### 変えてみる：bも動かす（for文を二重に）

上のセルの`1200`を、もう1つの`for`文の`b`に変えます。`for`文の中に`for`文を入れると、aとbのすべての組み合わせを試せます。`best`には (a, b, 二乗和) の3つをまとめて覚えます。`best[2]`は3つ目、つまり二乗和です。


In [ ]:
best = None
for a in range(5000, 8001, 100):
    for b in range(0, 2001, 100):          # b も動かす
        s = ((y - (a * x + b)) ** 2).sum()
        if best is None or s < best[2]:
            best = (a, b, s)
print(best)


31×21 ＝ 651通りを試して、a=6700・b=1200が選ばれました。


### TODO②：範囲を狭めて、刻みを細かくする

上の二重の`for`文のセルをコピーして、aを6400〜6800、bを1000〜1400の範囲で、どちらも10ずつ試すように変えてください。


In [ ]:
# TODO: 上のセルをコピーして、a を range(6400, 6801, 10)、

# 解答例②：範囲を狭めて、10 刻みで探す
best = None
for a in range(6400, 6801, 10):
    for b in range(1000, 1401, 10):
        s = ((y - (a * x + b)) ** 2).sum()
        if best is None or s < best[2]:
            best = (a, b, s)
print(best)


---


## 5. 谷底を、式一発で求める


総当たりは、刻みを細かくするほど時間がかかります。実は、二乗和の谷底は式一発で求められます。谷底では、aやbを少し動かしても二乗和は変わりません。つまり、$S(a, b)$をaで微分しても、bで微分しても0になります。

aで微分すると（$(y_i - a x_i - b)^2$をaで微分すると$-2x_i(y_i - a x_i - b)$になる、合成関数の微分）、

$$
\frac{\partial S}{\partial a} = \sum_{i=1}^{731} -2\,x_i\,\bigl(y_i - a\,x_i - b\bigr) = 0
$$

bで微分すると、

$$
\frac{\partial S}{\partial b} = \sum_{i=1}^{731} -2\,\bigl(y_i - a\,x_i - b\bigr) = 0
$$

です（$\partial$は、ほかの文字を止めて微分する、という印）。$-2$で割って、aとbを含む項を左に集めると、2本の式になります（$n = 731$）。

$$
\begin{aligned}
n\,b + \Bigl(\sum_i x_i\Bigr)\,a &= \sum_i y_i \\
\Bigl(\sum_i x_i\Bigr)\,b + \Bigl(\sum_i x_i^2\Bigr)\,a &= \sum_i x_i\,y_i
\end{aligned}
$$

知りたいのはaとbの2つで、式も2本です。連立方程式として解けます。


### 行列で書く：正規方程式

この2本の式は、行列で1本にまとめられます。台数を縦に並べた$\mathbf{y}$（731×1）、1の列と気温の列を並べた表$\mathbf{X}$（731×2）、切片と傾きを並べた$\boldsymbol{\beta} = (b, a)^\top$（2×1）を使うと、予測は$\mathbf{X}\boldsymbol{\beta}$で731日ぶんがまとめて出ます。そして、上の2本の式は

$$
\underbrace{\begin{pmatrix} n & \sum_i x_i \\ \sum_i x_i & \sum_i x_i^2 \end{pmatrix}}_{\mathbf{X}^\top\mathbf{X}}
\begin{pmatrix} b \\ a \end{pmatrix}
=
\underbrace{\begin{pmatrix} \sum_i y_i \\ \sum_i x_i\,y_i \end{pmatrix}}_{\mathbf{X}^\top\mathbf{y}}
$$

を書き下したものです。これを$\mathbf{X}^\top\mathbf{X}\boldsymbol{\beta} = \mathbf{X}^\top\mathbf{y}$と書き、正規方程式（normal equation）と呼びます。答えは

$$
\hat{\boldsymbol{\beta}} = \bigl(\mathbf{X}^\top\mathbf{X}\bigr)^{-1}\mathbf{X}^\top\mathbf{y}
$$

の1行です。$(\cdot)^{-1}$は逆行列（inverse matrix）で、掛けると1にあたる行列（単位行列）になる行列です。$\mathbf{X}$の列が何本になっても、同じ式で谷底が出ます。外れの二乗和を最小にして係数を決めるこのやり方を、最小二乗法と呼びます。


### お手本：正規方程式をコードにする

- 役割：正規方程式を、記号と1対1でコードにする
- 入力：配列`x`・`y`
- 出力：$\hat{\boldsymbol{\beta}}$（`beta`）と、切片`b_best`・傾き`a_best`

転置$\mathbf{X}^\top$は`X1.T`、行列の積は`@`、逆行列は`np.linalg.inv`です。`np.ones(731)`は1が731個並んだ配列で、`np.column_stack`は配列を列として横に並べ、表（行列）にします。


In [ ]:
X1 = np.column_stack([np.ones(len(x)), x])  # 1の列と、気温の列
A = X1.T @ X1                 # XᵀX
c = X1.T @ y                  # Xᵀy
beta = np.linalg.inv(A) @ c   # (XᵀX)⁻¹ Xᵀy
b_best, a_best = beta         # 切片と傾き
print(beta)


b=1214.64、a=6640.71。TODO②の総当たりの谷底（6650, 1210）のすぐ近くが、正確な谷底でした。


### 変えてみる：solveで解く

`np.linalg.solve(A, c)`は、逆行列を作らずに、連立方程式$\mathbf{A}\boldsymbol{\beta} = \mathbf{c}$を直接解きます。


In [ ]:
beta_s = np.linalg.solve(A, c)   # A β = c を、そのまま解く
print(beta_s)


答えは同じです。それでも実務では`solve`を使います。`inv`は逆行列を作る途中で、計算の誤差が広がりやすいからです。列がほぼ重なると、その誤差が大きくなります。逆行列の作りにくさの目安を条件数と呼びます。気温だけの$\mathbf{X}^\top\mathbf{X}$では約47ですが、気温と体感気温（相関0.992）を両方含む11本の列にすると、約31万に跳ね上がります。このデータでの`inv`と`solve`の差はまだ$10^{-9}$ほどとわずかですが、列が重なるほど広がります。


### fit() の中身も、同じ谷底

scikit-learnの`LinearRegression`も、この最小二乗法を中でやってくれる道具です。`fit`で学ばせ、`coef_`（傾き）と`intercept_`（切片）で結果を見ます。列は`df[['気温']]`のように二重のかっこ（表の形）で渡します。


In [ ]:
model_temp = LinearRegression()
model_temp.fit(df[['気温']], y)
print(model_temp.coef_[0])           # 傾き a
print(model_temp.intercept_)         # 切片 b


自分で書いた正規方程式の答えと一致しました。`fit()`の1行の中身は、いま自分で書いた計算と同じ谷底です。


### 変えてみる：R²を自分で計算する

当たり具合の物差しとして、R²（決定係数）を使います。「毎日平均と答えたときの二乗和（基準）のうち、直線でどれだけ減らせたか」の割合で、1に近いほどよく当たります。

$$
R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}
$$

分子が直線の二乗和、分母が毎日平均$\bar{y}$と答えたときの二乗和です。直線が平均より外れれば、分数が1を超えて、R²はマイナスになります。


In [ ]:
pred_line = a_best * x + b_best
sse_line = ((y - pred_line) ** 2).sum()      # 直線の二乗和
sst = ((y - y.mean()) ** 2).sum()            # 基準の二乗和
print(1 - sse_line / sst)                    # R²
print(model_temp.score(df[['気温']], y))     # 答え合わせ


R²は0.394。基準の二乗和のうち、約4割を気温の直線で減らせました。

実は、前回の気温帯だけで分けた段（外れ幅1211台）をR²で測ると0.439で、直線より少し上です。前回の散布図で見たとおり、台数は暑すぎると頭打ちになるので、まっすぐな直線では右端を外してしまうのです。それでも直線には、境目で跳ばず、1本の式で書けるという良さがあります。

手がかりを増やすこともできます。`df[['気温', '年']]`のように列を並べるだけで、気温と年を両方使う式（重回帰、multiple regression）になります。年を$x_{i2}$とすると、$\hat{y}_i = a_1 x_{i1} + a_2 x_{i2} + b$です。


In [ ]:
model_ty = LinearRegression()
model_ty.fit(df[['気温', '年']], y)
print(model_ty.coef_, model_ty.intercept_)
print(model_ty.score(df[['気温', '年']], y))


気温と年を使うと、R²は0.68まで上がりました。`coef_`に並ぶ各列に掛かる数（直線の傾きaにあたる数）を、係数と呼びます。


### 変えてみる：11本の列の表を作る

列を11本に増やしても、変わるのは$\mathbf{X}$の列の数だけです（731×12）。正規方程式のコードは、そのまま使えます。1の列に11本の列を並べた表を作ります。`df[cols].values`で、11列をまとめて配列にします。


In [ ]:
cols = ['季節', '年', '月', '祝日', '曜日', '平日',
        '天気', '気温', '体感気温', '湿度', '風速']
X11 = np.column_stack([np.ones(len(y)), df[cols].values])
print(X11.shape)


### TODO③：11本の列で正規方程式を解く

`X11`で正規方程式を作り、`np.linalg.solve`で$\hat{\boldsymbol{\beta}}$を求めて表示してください。出てくる係数は12個（切片1個と、11本の列の係数）です。


In [ ]:
# TODO: X11 で正規方程式を作り、np.linalg.solve で β を求める

# 解答例③：11本の列で、正規方程式を解く
beta11 = np.linalg.solve(X11.T @ X11, X11.T @ y)
print(beta11.round(2))


### 変えてみる：同じことを1行で

`LinearRegression`なら、1の列を自分で足してくれるので、`df[cols]`を渡すだけです。


In [ ]:
model11 = LinearRegression().fit(df[cols], y)
print(round(model11.intercept_, 2))
print(model11.coef_.round(2))


正規方程式を自分で書いた12個の係数と、1行の答えが一致します。中で何をしているかを知ったうえで、1行で済ませられる。これがライブラリの便利さです。


---


## 6. 自己採点は、甘くないか


ここまでのR²は、式を作るのに使った731日で測っていました。第1回の最後で見たとおり、作るのに使った日で採点すると、成績は甘く出ます。本当の実力は、まだ見ていない日で測る必要があります。

そこで、データを2つに分けます。式を作るのに使う訓練データ（過去問）と、最後まで取っておくテストデータ（本番）です。まず、第1回の段で試してみます。


In [ ]:
# @title 第1回の段を、作った日と、まだ見ていない日で採点する
d = df.copy()
d['気温帯'] = pd.cut(d['気温'] * 41, bins=[0, 10, 15, 20, 25, 30, 40])
tr, te = train_test_split(d, test_size=0.2, random_state=42)
for keys in [['気温帯', '年'], ['月', '年', '天気', '曜日']]:
    means = tr.groupby(keys, observed=True)['台数'].mean().rename('答え')
    pred_tr = tr.join(means, on=keys)['答え']
    pred_te = te.join(means, on=keys)['答え']
    ok = pred_te.notna()             # 訓練に同じ組み合わせがあり、答えを出せた日
    err_tr = np.abs(tr['台数'] - pred_tr)
    err_te = np.abs(te['台数'] - pred_te)
    print(keys)
    print('  作った日          :', round(err_tr.mean()), '台')
    print('  答えを出せない日  :', int((~ok).sum()), '日')
    print('  答えを出せた日    :', round(err_te[ok].mean()), '台')


気温帯×年の段は、作った日でも見ていない日でも、外れ幅は770台ほどで変わりません（訓練の584日だけで段を作り直したので、731日すべてで作った前回の762台とは少し違います）。

### 考えてみよう：細かい段は、どうなる？

月×年×天気×曜日の細かい段は、作った日では393台しか外れません。上の出力の下半分を見る前に、まだ見ていない147日で試すと何が起きそうかを予想して、下に書いてください。


（ここに予想を書く）


テスト147日のうち45日は、訓練に同じ組み合わせ（同じ月・年・天気・曜日）の日が1日もありません。その日には、段の答えそのものが出せないのです。答えを出せた102日だけで見ても、外れ幅は665台あり、作った日の393台よりずっと大きくなります。細かく分けた段は、作った日の台数を覚えていただけで、新しい組み合わせの日には答えを持っていなかったのです。（`~ok`の`~`は、TrueとFalseを入れかえる記号です。）


### お手本：データを訓練とテストに分ける

- 役割：11本の列を手がかりに、データを訓練8割・テスト2割に分ける
- 入力：手がかりの表`X`と台数`y`
- 出力：`X_tr`・`X_te`・`y_tr`・`y_te`

`train_test_split`は、行をばらばらに混ぜてから分ける関数です。`test_size=0.2`でテストを2割に、`random_state=42`で混ぜ方を固定します（何度やっても同じ分け方になる）。


In [ ]:
cols = ['季節', '年', '月', '祝日', '曜日', '平日',
        '天気', '気温', '体感気温', '湿度', '風速']
X = df[cols].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                          random_state=42)
print(len(X_tr), len(X_te))


訓練が584日、テストが147日になりました。まだ見ていないテストでの当たり具合を、汎化性能（generalization performance）と呼びます。


### TODO④：訓練で学び、訓練とテストのR²を比べる

`LinearRegression`を訓練データだけで`fit`し、`score`で訓練データのR²とテストデータのR²を表示してください。


In [ ]:
# TODO: 訓練データだけで fit し、訓練の R² とテストの R² を表示する

# 解答例④：訓練で学び、訓練とテストの R² を比べる
model = LinearRegression().fit(X_tr, y_tr)
print('訓練 R²  :', model.score(X_tr, y_tr))
print('テスト R²:', model.score(X_te, y_te))


訓練のR²は0.791、テストのR²は0.828です。2つがほぼ同じです（テストがたまたま少し高い）。つまり、この直線の式は作った日を覚えすぎておらず、まだ見ていない日にも同じくらい当たるということです。汎化性能は、訓練並みです。


### 考えてみよう：完璧すぎる式

11本の列に、ある2列を足すと、まだ見ていないテストのR²がちょうど1.0になります。どの日もぴったり当たるのです。どの2列で、なぜそうなるのか、どこがおかしいのかを、導入の「手元のデータ」を思い出して予想し、下に書いてください。


（ここに予想を書く）


### 変えてみる：リークを再現する

第1回で、`一般客`と`会員`の列を「足すとちょうど台数になる内訳なので、残すと答えが最初から分かってしまう」として落としました。わざとこの2列を手がかりに足すと、どうなるでしょうか。上のお手本の`cols`に`+`で2列をつなぎ、分け方も同じにします。


In [ ]:
cols_leak = cols + ['一般客', '会員']
X_leak = df[cols_leak].values
XL_tr, XL_te, yL_tr, yL_te = train_test_split(
    X_leak, y, test_size=0.2, random_state=42)
model_leak = LinearRegression().fit(XL_tr, yL_tr)
print('テスト R²:', model_leak.score(XL_te, yL_te))


まだ見ていないテストでも、R²は1.0。どの日もぴったり当たりました。台数＝一般客＋会員なので、一般客と会員の係数を1、ほかを0にすれば、外れは0になります。答えそのものを手がかりに入れていたのです。このように、答え（やそれに近い情報）が手がかりに漏れ込むことを、リーク（leakage、情報の漏れ）と呼びます。訓練とテストに分けても、両方に同じ漏れがあるので見抜けません。

明日の一般客と会員の数は、明日が来るまで分かりません。この式は、明日の台数を当てるのには使えないのです。テストの成績が良すぎるときは、まずリークを疑います。


---


## 7. 式を曲げすぎると？


6節で、直線の式は訓練とテストでほぼ同じくらい当たると分かりました。直線が暑い日の頭打ちを外すなら、式を曲げればよいのでは？ 気温の2乗、3乗……を足していくと、式は曲線になります（多項式回帰）。$\hat{y} = b + a_1 x + a_2 x^2 + \cdots + a_d x^d$の、何乗まで使うか（$d$）を次数と呼びます。曲げるほど、訓練データにはよく当たるはずです。では、テストデータには？


### お手本：3次の曲線を当てはめる

- 役割：気温だけを手がかりに、3次の曲線を訓練データで学び、訓練とテストのR²を出す
- 入力：訓練とテストの気温・台数
- 出力：2つのR²

`X_tr[:, [7]]`の`:`はすべての行、`[7]`は8本目の列（番号は0から）を表します。`[7]`とかっこで囲むと、1列の表の形のまま取り出せます。`fit`は手がかりを表の形で受け取るので、この形にしておきます。

`PolynomialFeatures(3)`は気温から2乗・3乗の列を作る部品、`Pipeline`は部品を順につなぐ道具です。


In [ ]:
x1_tr = X_tr[:, [7]]    # 8本目の列（気温）だけ
x1_te = X_te[:, [7]]
model3 = Pipeline([('poly', PolynomialFeatures(3)),
                   ('lr', LinearRegression())])
model3.fit(x1_tr, y_tr)
print(model3.score(x1_tr, y_tr), model3.score(x1_te, y_te))


In [ ]:
# @title 図：次数を変えた曲線（横軸は℃）
fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=True)
x_plot = np.linspace(x1_tr.min(), x1_tr.max(), 200).reshape(-1, 1)
for ax, deg in zip(axes, [1, 3, 8, 15]):
    m = Pipeline([('poly', PolynomialFeatures(deg)), ('lr', LinearRegression())]).fit(x1_tr, y_tr)
    ax.scatter(x1_tr * 41, y_tr, s=6, alpha=0.2, color='#0066cc')
    ax.plot(x_plot * 41, np.clip(m.predict(x_plot), -2000, 11000), color='#e63946', lw=2)
    ax.set_title(f'{deg}次'); ax.set_ylim(0, 9000); ax.set_xlabel('気温（℃）')
plt.tight_layout(); plt.show()


### TODO⑤：次数を変えて、訓練とテストのR²を並べる

`for`文で次数を1・3・8・15と変えながら、お手本のセルと同じことをして、次数ごとの訓練のR²とテストのR²を表示してください。


In [ ]:
# TODO: for deg in [1, 3, 8, 15]: として、次数ごとの

# 解答例⑤：次数ごとの訓練 R² とテスト R²
for deg in [1, 3, 8, 15]:
    m = Pipeline([('poly', PolynomialFeatures(deg)),
                  ('lr', LinearRegression())])
    m.fit(x1_tr, y_tr)
    print(deg, '次:', round(m.score(x1_tr, y_tr), 4),
          round(m.score(x1_te, y_te), 4))


訓練のR²は次数とともに上がっていきます（0.388 → 0.472 → 0.476 → 0.481）。テストのR²は0.404 → 0.414 → 0.408 → 0.407で、3次がいちばん高いものの、ほぼ横ばいです。訓練が584日もあると、曲線は1日1日の偶然の揺れに振り回されにくく、曲げすぎてもテストの成績はほとんど崩れません。

では、データが少なかったらどうでしょう。


### 考えてみよう：20日だけなら？

訓練を584日から20日に減らし、テストは同じ147日のままで、1次・3次・8次と曲げていきます。訓練のR²とテストのR²は、それぞれどう動くと思いますか。予想を下に書いてから、次のセルを実行してください。


（ここに予想を書く）


### 変えてみる：訓練を20日だけにする

- 役割：訓練データを最初の20日だけにして、1・3・8次の曲線を学び、訓練とテストのR²を出す
- 入力：訓練の最初の20日（`x1_tr[:20]`・`y_tr[:20]`）と、テストの147日
- 出力：次数ごとの、訓練のR²とテストのR²

`[:20]`は、3節で見たスライスです。番号0〜19の20日を取り出します。テストは同じ147日のままです。


In [ ]:
x20 = x1_tr[:20]     # 訓練の最初の20日だけ
y20 = y_tr[:20]
for deg in [1, 3, 8]:
    m = Pipeline([('poly', PolynomialFeatures(deg)),
                  ('lr', LinearRegression())])
    m.fit(x20, y20)
    print(deg, '次:', round(m.score(x20, y20), 3),
          round(m.score(x1_te, y_te), 3))


In [ ]:
# @title 図：20日で学んだ1次と8次の曲線
x_plot = np.linspace(x1_te.min(), x1_te.max(), 400).reshape(-1, 1)
plt.scatter(x1_te * 41, y_te, s=10, alpha=0.35, color='#888888', label='テスト（147日）')
plt.scatter(x20 * 41, y20, s=45, color='#0066cc', label='訓練（20日）')
for deg, c in [(1, '#2a9d8f'), (8, '#e63946')]:
    m = Pipeline([('poly', PolynomialFeatures(deg)), ('lr', LinearRegression())]).fit(x20, y20)
    plt.plot(x_plot * 41, m.predict(x_plot), color=c, lw=2.5, label=f'{deg}次')
plt.xlim(x1_te.min() * 41, x1_te.max() * 41); plt.ylim(0, 9000)
plt.xlabel('気温（℃）'); plt.ylabel('利用台数'); plt.legend(loc='upper left')
plt.show()


訓練のR²は0.228 → 0.444 → 0.499と、曲げるほど上がります。ところがテストのR²は0.362 → −0.288 → −28.353と、マイナスに落ちました。R²のマイナスは、毎日平均と答えるより悪いということです。図を見ると、8次の曲線は青い20日に合わせて曲がり、20日の外（8℃未満や29℃超）では大きく外れています。

曲げすぎた式は、訓練データの偶然の揺れまで覚えてしまいます。これが過学習です。データが多いと目立ちませんが、少ないほどはっきり出ます。


---


## 8. 明日は何台？


6節で、11本の列の式は、まだ見ていない日にも同じくらい当たると分かりました。ただ、明日の湿度や風速は、まだ分かりません。明日に答えるのは、5節の気温と年の2つを使った式にします。まず、この式も6節と同じ分け方で測ります。6節のお手本の`X`を、2本の列の`Xty`に変えただけです。


In [ ]:
Xty = df[['気温', '年']].values
Xty_tr, Xty_te, yty_tr, yty_te = train_test_split(
    Xty, y, test_size=0.2, random_state=42)
model_ty = LinearRegression().fit(Xty_tr, yty_tr)
print('訓練 R²  :', model_ty.score(Xty_tr, yty_tr))
print('テスト R²:', model_ty.score(Xty_te, yty_te))


テストのR²は0.713で、11本の列の0.828より低くなりました。手がかりを減らしたぶん、外れます。その代わり、明日に使える式になりました。

次に、作った日と、まだ見ていない日の外れ幅の平均$\frac{1}{n}\sum_i |y_i - \hat{y}_i|$を測ります。明日が27℃・2年目なら、何台でしょうか。


In [ ]:
err_tr = np.abs(yty_tr - model_ty.predict(Xty_tr)).mean()
err_te = np.abs(yty_te - model_ty.predict(Xty_te)).mean()
print('作った日の外れ幅  :', round(err_tr), '台')
print('まだ見ていない日  :', round(err_te), '台')
pred_t = model_ty.predict([[27 / 41, 1]])[0]     # 27℃・2年目
print('明日（27℃・2年目）:', round(pred_t), '台')


`predict`に手がかりの表を渡すと、予測が返ります。`[[27 / 41, 1]]`は1行の表で、気温は41で割った値、年は2年目なので1です。式に代入しているだけで、$\hat{y} = a_1 \cdot \frac{27}{41} + a_2 \cdot 1 + b$を計算しています。予測は並びで返るので、`[0]`で1つ目を取り出します。


答えは「およそ6600台。まだ見ていない日での外れ幅は、平均840台くらい」です。

第1回の答え（およそ7200台）より低いのは、半分ほどが天気の違いです。第1回は25〜30℃・2年目の晴れの日（54日）だけの平均でした。今日の式は天気を入れていないので、晴れも雨も混ぜた見積もりになっています。25〜30℃・2年目の84日を、天気を混ぜて平均すると6843台で、晴れの54日の7198台より355台少なくなります。差の633台の残り278台は、直線が頭打ちの曲がり方をなぞれないぶんです。

外れ幅も、同じ物差しで比べる必要があります。第1回の「500台前後」は、晴れの54日の中で、作った日で測った外れ幅でした。同じテスト147日で比べると、気温帯×年の段（6節）は773台、今日の直線は837台です。直線は境目で跳ばない代わりに、暑い日の頭打ちを外すぶん、少し外れます。

まだ言えないことも残っています。

- 1回の分け方の運：テストのR²（6節の0.828、この節の0.713）は、`random_state=42`で分けた1回の値です。分け方を変えたら、どれくらい動くのでしょうか
- データを整えていない：今日のデータは、欠けた値も文字の列もない、きれいなデータでした。実務のデータはそうはいきません
- 物差しは何がよいか：R²も二乗和を使うので、大外れを重く見ている。外れ幅の平均はどの外れも同じ重さ。どちらで測るかは場面しだい
- 係数をどう読むか：5節の式の年の係数2083は「2年目は同じ気温でも2083台多い」と読めるのか


---


## 目標に答えられたか


- 今日の目標は「1本の式を立て、まだ来ていない日にどれだけ当たるかを測る」ことでした。8節の答えは何台で、まだ見ていない日での外れ幅は何台でしたか？
- 4節の総当たりと5節の式一発で、谷底のa・bはどれくらい近かったでしょうか？`LinearRegression`の答えとは一致しましたか？
- TODO④で、訓練のR²とテストのR²はどれくらい違いましたか？6節の細かい段と比べて、何が違いますか？
- TODO⑤で、次数を上げると訓練とテストのR²はどう動きましたか？


---


## 今日の要点


- 今日、持ち帰る1文：作るのに使ったデータの成績は信じない。まだ見ていないデータで測り、良すぎたら漏れを疑う
- 二乗和を最小にする係数は、正規方程式$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$で求まる。総当たり（`for`・`if`）、正規方程式（`@`・`.T`・`solve`）、`LinearRegression`の1行は、同じ谷底にたどり着く
- 直線の良さは、外れの二乗和で測る。二乗和を最小にするa・bを選ぶのが最小二乗法で、`LinearRegression`の中身そのもの
- `for`文で1日ずつ足した二乗和と、numpyのまとめて計算は一致する。総当たりは`for`文と`if`文（小さければ覚え直す）で書ける
- R²は、毎日平均と答える基準から、二乗和をどれだけ減らせたかの割合
- 作るのに使ったデータで測った成績は甘い。訓練とテストに分けて、まだ見ていないデータで測る（汎化性能）
- 答えの内訳を手がかりに入れると、テストでも完璧に当たってしまう（リーク）。良すぎる成績は疑う
- 式を曲げすぎると、訓練には当たるがテストには当たらない（過学習）。データが少ないほど、はっきり出る


---


## 次回へ


今日は、段を1本の式にし、まだ見ていない日で実力を測りました。ただ、今日のデータは最初からきれいに整っていました。

次回は、場面を民泊の値付けに変えます。極端な値、欠けた値、文字の列、目盛りのばらばらな列。実務のデータにつきもののこうした「汚れ」を一つずつ整えていきます。自分の部屋に、1泊いくらの値段を付ければいいでしょうか。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

6節では`random_state=42`で1回だけ分けて、テストのR²を測りました。今度は分け方を変えて、同じことを5回くり返します。

`for`文で`random_state`を0, 1, 2, 3, 4と変えながら、11本の列の`X`と`y`を訓練8割・テスト2割に分けてください。それぞれの分け方で`LinearRegression`を訓練データで学び、テストのR²を5つ表示してください。5つの値は、あとで使うのでリスト`r2_list`にためておきます（`r2_list.append(値)`で1つ足せます）。


In [ ]:
r2_list = []
for rs in [0, 1, 2, 3, 4]:
    X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
        X, y, test_size=0.2, random_state=rs)
    model_r = LinearRegression().fit(X_tr_r, y_tr_r)
    r2 = model_r.score(X_te_r, y_te_r)
    r2_list.append(r2)
    print('random_state =', rs, ' テスト R² =', round(r2, 4))


<details><summary>詰まったら</summary>

`r2_list = []`から始め、`for rs in [0, 1, 2, 3, 4]:`の中で
`train_test_split(X, y, test_size=0.2, random_state=rs)` → `LinearRegression().fit(…)` → `.score(X_te_r, y_te_r)`の順に書きます。

</details>


### 応用②（判断）

応用①の5つのテストのR²のうち、最大と最小の差はいくつですか。丸めていない値で差を計算し、小数第2位に丸めて答えてください（例: 0.12）。


In [ ]:
r2_arr = np.array(r2_list)
diff = r2_arr.max() - r2_arr.min()
print('最大', round(r2_arr.max(), 4),
      '− 最小', round(r2_arr.min(), 4), '=', round(diff, 4))
print('答え:', f'{diff:.2f}')


<details><summary>詰まったら</summary>

リストを`np.array(r2_list)`で配列にすると、`.max()`と`.min()`が使えます。

</details>


### 応用③（解釈）

6節では、1回の分け方でテストのR²が0.828でした。この1回の値だけを根拠に「この式の実力はR²＝0.83」と報告してよいでしょうか。応用①の5つの値の幅を根拠に、自転車シェアの運営担当者に向けて3行程度で書いてください。


（模範例）

テストのR²は、分け方を変えるだけで0.75〜0.83まで動き、最大と最小で0.07の差がありました。

6節の0.828は5回のどれよりも高く、たまたま当てやすい日がテストに入った分け方だった可能性があります。

報告では、1回の値で言い切らず「R²はおよそ0.75〜0.83。分け方によって0.07ほど動く」と、幅を付けて伝えるのがよいと考えます。


---


## 発展（任意）


### 日付順に分ける：未来を覗かない

6節の`train_test_split`は、行をばらばらに混ぜてから分けました。でも、自転車シェアの台数は日付順に並んだデータです。混ぜて分けると、2012年の日で学んで2011年の日を当てるという「未来を覗いた」採点が混ざります。

`shuffle=False`を付けると、行を混ぜずに上から順に分けます。表は日付順なので、前の8割で学び、後ろの2割を当てることになります。運営者が本当にやりたいのは、昨日までのデータで明日を当てることなので、こちらのほうが実際に近い採点です。


In [ ]:
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X, y, test_size=0.2,
                                              shuffle=False)
n = len(y_tr2)
print('訓練  :', df['日付'].iloc[0], '〜', df['日付'].iloc[n - 1])
print('テスト:', df['日付'].iloc[n], '〜', df['日付'].iloc[-1])
m2 = LinearRegression().fit(X_tr2, y_tr2)
print('テスト R²:', round(m2.score(X_te2, y_te2), 3))


In [ ]:
m1 = LinearRegression().fit(X_tr, y_tr)           # 6節の式
err1 = np.abs(y_te - m1.predict(X_te)).mean()       # 混ぜて分けた
err2 = np.abs(y_te2 - m2.predict(X_te2)).mean()     # 日付順に分けた
print('混ぜて分けた:', round(err1), '台')
print('日付順      :', round(err2), '台')


読み方　日付順に分けると、テストのR²は0.828から0.613に下がり、外れ幅は617台から864台に増えます。テストは2012年8月7日から年末までで、2年目の9〜12月は訓練に1日も入っていません。秋から冬にかけての台数の動きは、1年目の秋冬からしか学べていないのです。ただし、テストの日も6節とは違います。この差には、分け方の違いと採点する日の違いの両方が入っています。

混ぜて分けた0.828は、「過去と未来が混ざった、少し甘い採点」だったことになります。どちらの採点が正しいかは、何に使う式かで決まります。明日を当てる式なら、日付順に分けたほうが正直です。

試すなら、`test_size`を0.1や0.3に変えて、テストのR²がどう動くか見てください。
